# Curve Fitting Demonstrations

This notebook showcases all capabilities of the `lab_math_tools.curve_fitting` module:
- **`fit_metrics`**: A standalone utility that computes goodness-of-fit statistics (MSE, RMSE, MAE, R², adjusted R², residual std) from any pair of true and predicted arrays.
- **`fit_linear`**: Ordinary least-squares linear regression with a full metrics report and parameter covariance matrix.
- **`fit_polynomial`**: Polynomial regression of arbitrary degree, with optional `zero_degrees` to pin specific power coefficients to zero (e.g., forcing a fit to `ax³ + cx` without constant or quadratic terms).
- **`fit_preset`**: Non-linear least-squares fitting to five built-in model shapes — sine, exponential, logarithm, power law, and Gaussian — each with an optional `bias` flag to control the additive constant.
- **`fit_custom`**: Fitting to any user-defined function that follows the project naming conventions (`s_f(v_x)`), where `v_x[0]` is the x-data, optional `v_fixed_params` populate the next slots, and the remaining entries are free parameters found by the optimizer.
- **`FitResult`**: The unified return type for every fitting function, carrying parameters, covariance, residuals, metrics, and helper methods `.summary()` and `.plot_residuals()`.


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from lab_math_tools.curve_fitting import (
    fit_metrics,
    fit_linear,
    fit_polynomial,
    fit_preset,
    fit_custom,
    FitResult,
)

RNG = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 120


---
## 1. `FitResult` — The Unified Return Type

Every fitting function returns a `FitResult` dataclass. Understanding its fields and helpers is the best starting point.

| Field | Type | Description |
|---|---|---|
| `v_params` | `np.ndarray` | Best-fit parameter values |
| `m_covariance` | `np.ndarray` | Parameter covariance matrix |
| `v_residuals` | `np.ndarray` | Residuals: `y − ŷ` at each sample |
| `metrics` | `dict[str, float]` | Goodness-of-fit statistics |
| `model_name` | `str` | Human-readable model label |

Helper methods:
- `.summary()` → formatted string with parameters and metrics.
- `.plot_residuals(ax=None)` → stem plot of residuals; follows the module's OOP pattern (accepts and returns `ax`).


In [ ]:
# Quick demonstration using fit_linear so we have a FitResult to inspect
v_x = np.linspace(0, 10, 50)
v_y = 3.0 * v_x + 5.0 + RNG.normal(0, 1.5, 50)

result = fit_linear(v_x, v_y)

# .summary() prints a readable overview
print(result.summary())


In [ ]:
# .plot_residuals() returns a matplotlib Axes (OOP style)
fig, ax = plt.subplots(figsize=(8, 3))
result.plot_residuals(ax=ax)
plt.tight_layout()
plt.show()


---
## 2. `fit_metrics` — Standalone Goodness-of-Fit Metrics

`fit_metrics(v_y, v_y_pred, n_params=0)` computes statistics purely from true and predicted arrays.  
Use it whenever you have predictions from your own model and want a standard metric report.

| Metric key | Formula | Notes |
|---|---|---|
| `mse` | mean((y − ŷ)²) | Mean Squared Error |
| `rmse` | √MSE | Root Mean Squared Error |
| `mae` | mean(\|y − ŷ\|) | Mean Absolute Error |
| `r2` | 1 − SS_res / SS_tot | Coefficient of Determination |
| `std_residuals` | std(y − ŷ) | Spread of residuals |
| `r2_adj` | adjusted R² | Only when `n_params > 0` |


In [ ]:
# Perfect fit
v_y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print("Perfect fit metrics:")
print(fit_metrics(v_y_true, v_y_true, n_params=1))
print()

# Imperfect fit — shifted by 0.5
v_y_pred = v_y_true + 0.5
print("Shifted-by-0.5 metrics:")
print(fit_metrics(v_y_true, v_y_pred, n_params=1))


---
## 3. `fit_linear` — Linear Regression

Fits `y = a₀ + a₁·x` using ordinary least squares.

```python
result = fit_linear(v_x, v_y)
# result.v_params = [intercept, slope]
```

The full metric set including `r2_adj` is always computed.  
The `m_covariance` matrix is the 2×2 parameter covariance, estimated as `σ² · (XᵀX)⁻¹`.


### 3.1 Perfect Line Recovery


In [ ]:
v_x = np.linspace(0, 10, 100)
v_y = 3.5 * v_x - 2.0   # slope=3.5, intercept=-2.0

result = fit_linear(v_x, v_y)
print(result.summary())


### 3.2 Fitting Noisy Experimental Data


In [ ]:
v_x = np.linspace(0, 10, 60)
v_y = 2.0 * v_x + 4.0 + RNG.normal(0, 1.5, 60)

result = fit_linear(v_x, v_y)
intercept, slope = result.v_params

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Data + fit
ax = axes[0]
ax.scatter(v_x, v_y, s=20, alpha=0.6, label="Noisy data")
ax.plot(v_x, intercept + slope * v_x, color="red", lw=2, label=f"Fit: y = {slope:.2f}x + {intercept:.2f}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Linear Fit  (R² = {result.metrics['r2']:.4f})")
ax.legend()

# Residuals
result.plot_residuals(ax=axes[1])

plt.tight_layout()
plt.show()

print("\nCovariance matrix:")
print(result.m_covariance)


---
## 4. `fit_polynomial` — Polynomial Regression

Fits `y = a₀ + a₁·x + … + aₙ·xⁿ` using least squares.

```python
result = fit_polynomial(v_x, v_y, degree, zero_degrees=None)
# result.v_params = [a0, a1, ..., a_degree]  (ascending power order)
```

The `zero_degrees` parameter pins specific power coefficients to exactly 0,  
enabling fits like `f(x) = a₁x + a₃x³` (pass `zero_degrees=[0, 2]`).


### 4.1 Standard Polynomial Fit (degree 2)


In [ ]:
v_x = np.linspace(-3, 3, 150)
v_y = 2.0 * v_x**2 + 3.0 * v_x + 1.0 + RNG.normal(0, 0.5, 150)

result = fit_polynomial(v_x, v_y, degree=2)
a0, a1, a2 = result.v_params
print(result.summary())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(v_x, v_y, s=15, alpha=0.5, label="Data")
v_fit = a0 + a1 * v_x + a2 * v_x**2
ax.plot(v_x, v_fit, color="red", lw=2,
        label=f"Fit: y = {a2:.2f}x² + {a1:.2f}x + {a0:.2f}")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"Degree-2 Polynomial Fit  (R² = {result.metrics['r2']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()


### 4.2 Degree-3 Polynomial (perfect data)


In [ ]:
v_x = np.linspace(-2, 2, 200)
v_y = v_x**3 - v_x   # true: a0=0, a1=-1, a2=0, a3=1

result = fit_polynomial(v_x, v_y, degree=3)
print("Recovered coefficients [a0, a1, a2, a3]:")
print(result.v_params.round(6))


### 4.3 `zero_degrees` — Constrained Polynomial Fit

Physics often dictates that certain powers must vanish (e.g., a spring with no rest displacement and no quadratic term).  
Pass `zero_degrees=[0, 2]` to pin `a₀ = a₂ = 0` and fit only `a₁` and `a₃`.


In [ ]:
v_x = np.linspace(-2, 2, 200)
# True model: f(x) = 1.5·x³ + 2·x  (no constant, no quadratic)
v_y = 1.5 * v_x**3 + 2.0 * v_x + RNG.normal(0, 0.1, 200)

# Unconstrained degree-3 fit
r_unconstrained = fit_polynomial(v_x, v_y, degree=3)

# Constrained: force a0 = a2 = 0
r_constrained = fit_polynomial(v_x, v_y, degree=3, zero_degrees=[0, 2])

print("Unconstrained coefficients [a0, a1, a2, a3]:")
print(r_unconstrained.v_params.round(4))
print()
print("Constrained coefficients [a0=0, a1, a2=0, a3]:")
print(r_constrained.v_params.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(v_x, v_y, s=15, alpha=0.4, label="Data")
ax.plot(v_x, np.polyval(r_unconstrained.v_params[::-1], v_x),
        lw=2, label="Unconstrained fit")
ax.plot(v_x, np.polyval(r_constrained.v_params[::-1], v_x),
        lw=2, linestyle="--", label="Constrained (a0=a2=0)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Constrained Polynomial Fit with zero_degrees")
ax.legend()
plt.tight_layout()
plt.show()


---
## 5. `fit_preset` — Built-in Model Shapes

Fits data to one of five pre-defined models using `scipy.optimize.curve_fit` (non-linear least squares).

```python
result = fit_preset(v_x, v_y, model, bias=True, v_p0=None, bounds=(-np.inf, np.inf))
```

| `model` | Formula (`bias=True`) | `v_params` |
|---|---|---|
| `"sine"` | A·sin(ω·x + φ) + C | [A, ω, φ, C] |
| `"exponential"` | A·exp(b·x) + C | [A, b, C] |
| `"logarithm"` | A·ln(b·x) + C | [A, b, C] |
| `"power"` | A·x^b | [A, b] |
| `"gaussian"` | A·exp(−(x−μ)²/2σ²) | [A, μ, σ] |

`bias=False` removes the additive constant C (not applicable to `power` and `gaussian`).


### 5.1 Sine Wave


In [ ]:
# True parameters: A=2.5, omega=1.0, phi=0.3, C=1.0
v_x = np.linspace(0, 4 * np.pi, 200)
A, omega, phi, C = 2.5, 1.0, 0.3, 1.0
v_y = A * np.sin(omega * v_x + phi) + C + RNG.normal(0, 0.15, 200)

result = fit_preset(v_x, v_y, model="sine", v_p0=np.array([2.5, 1.0, 0.3, 1.0]))
A_fit, omega_fit, phi_fit, C_fit = result.v_params
print(result.summary())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(v_x, v_y, s=10, alpha=0.5, label="Data")
axes[0].plot(v_x, A_fit * np.sin(omega_fit * v_x + phi_fit) + C_fit,
             color="red", lw=2, label="Sine fit")
axes[0].set_title(f"Sine Fit  (R² = {result.metrics['r2']:.4f})")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].legend()
result.plot_residuals(ax=axes[1])
plt.tight_layout()
plt.show()


### 5.2 Exponential


In [ ]:
v_x = np.linspace(0, 3, 150)
A, b, C = 3.0, 0.8, 1.0
v_y = A * np.exp(b * v_x) + C + RNG.normal(0, 0.3, 150)

result = fit_preset(v_x, v_y, model="exponential", v_p0=np.array([3.0, 0.8, 1.0]))
A_fit, b_fit, C_fit = result.v_params
print(result.summary())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(v_x, v_y, s=20, alpha=0.5, label="Data")
ax.plot(v_x, A_fit * np.exp(b_fit * v_x) + C_fit,
        color="red", lw=2, label=f"Fit: {A_fit:.2f}·exp({b_fit:.2f}·x) + {C_fit:.2f}")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"Exponential Fit  (R² = {result.metrics['r2']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()


### 5.3 Logarithm


In [ ]:
v_x = np.linspace(0.5, 8, 150)
A, b, C = 4.0, 1.0, 0.5
v_y = A * np.log(b * v_x) + C + RNG.normal(0, 0.2, 150)

result = fit_preset(v_x, v_y, model="logarithm", v_p0=np.array([4.0, 1.0, 0.5]))
A_fit, b_fit, C_fit = result.v_params
print(result.summary())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(v_x, v_y, s=20, alpha=0.5, label="Data")
ax.plot(v_x, A_fit * np.log(b_fit * v_x) + C_fit,
        color="red", lw=2, label=f"Fit: {A_fit:.2f}·ln({b_fit:.2f}·x) + {C_fit:.2f}")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"Logarithm Fit  (R² = {result.metrics['r2']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()


### 5.4 Power Law


In [ ]:
v_x = np.linspace(0.1, 6, 150)
A, b = 2.0, 1.5
v_y = A * v_x**b + RNG.normal(0, 0.4, 150)

result = fit_preset(v_x, v_y, model="power", v_p0=np.array([2.0, 1.5]))
A_fit, b_fit = result.v_params
print(result.summary())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(v_x, v_y, s=20, alpha=0.5, label="Data")
ax.plot(v_x, A_fit * v_x**b_fit, color="red", lw=2,
        label=f"Fit: {A_fit:.2f}·x^{b_fit:.2f}")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"Power Law Fit  (R² = {result.metrics['r2']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()


### 5.5 Gaussian


In [ ]:
v_x = np.linspace(-6, 6, 250)
A, mu, sigma = 5.0, 1.0, 1.5
v_y = A * np.exp(-((v_x - mu)**2) / (2 * sigma**2)) + RNG.normal(0, 0.1, 250)

result = fit_preset(v_x, v_y, model="gaussian", v_p0=np.array([5.0, 1.0, 1.5]))
A_fit, mu_fit, sigma_fit = result.v_params
print(result.summary())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(v_x, v_y, s=15, alpha=0.4, label="Data")
ax.plot(v_x, A_fit * np.exp(-((v_x - mu_fit)**2) / (2 * sigma_fit**2)),
        color="red", lw=2,
        label=f"Fit: A={A_fit:.2f}, μ={mu_fit:.2f}, σ={abs(sigma_fit):.2f}")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"Gaussian Fit  (R² = {result.metrics['r2']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()


### 5.6 `bias=False` — Removing the Additive Constant

When `bias=False`, the constant offset C is fixed to zero and excluded from `v_params`.  
This is useful when the physics of the problem guarantees a zero baseline.


In [ ]:
v_x = np.linspace(0, 2 * np.pi, 150)
v_y = 3.0 * np.sin(2.0 * v_x + 0.5)   # C = 0 by construction

# With bias=True  → 4 parameters
r_bias = fit_preset(v_x, v_y, model="sine", v_p0=np.array([3.0, 2.0, 0.5, 0.0]))
# With bias=False → 3 parameters
r_no_bias = fit_preset(v_x, v_y, model="sine", bias=False, v_p0=np.array([3.0, 2.0, 0.5]))

print("bias=True  → v_params:", r_bias.v_params.round(4))
print("bias=False → v_params:", r_no_bias.v_params.round(4))
print(f"model_name with bias=False: '{r_no_bias.model_name}'")


---
## 6. `fit_custom` — User-Defined Function Fitting

Fits data to any function you define, following the project's naming conventions.

```python
result = fit_custom(v_x, v_y, f, v_p0, v_fixed_params=None, bounds=(-np.inf, np.inf))
```

**Vector layout inside `f` per sample:**

```
v_x[0]               → x data for this sample
v_x[1 : 1+n_fixed]   → v_fixed_params  (pre-set constants)
v_x[1+n_fixed : ...]  → free parameters (varied by optimizer)
```

- **`v_p0`**: Initial guess for the **free** parameters only.  
- **`v_fixed_params`**: Constant values you supply (not fitted). `None` → no fixed params.  
- **`result.v_params`**: Contains only the **fitted** (free) parameters.


### 6.1 Simple Scaling — One Free Parameter

The simplest case: a model with a single scale factor.


In [ ]:
# Model: f(x) = slope * x
# v_x[0] = x data,  v_x[1] = slope (free)
def s_f(v2_x: np.ndarray) -> float:
    return v2_x[1] * v2_x[0]

v_x = np.linspace(0, 5, 100)
v_y = 3.5 * v_x + RNG.normal(0, 0.2, 100)

result = fit_custom(v_x, v_y, s_f, v_p0=np.array([1.0]))
print(f"Fitted slope: {result.v_params[0]:.4f}  (true: 3.5)")
print(result.summary())


### 6.2 Damped Oscillation — Two Free Parameters

A more physically meaningful example: fitting a damped sine wave.

$$f(x) = A \cdot e^{-\gamma x} \cdot \sin(\omega x)$$

Here the angular frequency `ω` is known from theory, so it is passed as a **fixed parameter**.  
Only amplitude `A` and damping coefficient `γ` need to be fitted.


In [ ]:
# Model: A * exp(-gamma * x) * sin(omega * x)
# v_x[0] = x data
# v_x[1] = omega (FIXED)
# v_x[2] = A     (free)
# v_x[3] = gamma (free)
def s_f_damped(v4_x: np.ndarray) -> float:
    x, omega, A, gamma = v4_x
    return A * np.exp(-gamma * x) * np.sin(omega * x)

# True parameters
TRUE_OMEGA = 2.5   # fixed — known from theory
TRUE_A     = 4.0
TRUE_GAMMA = 0.4

v_x = np.linspace(0, 5, 300)
v_y = TRUE_A * np.exp(-TRUE_GAMMA * v_x) * np.sin(TRUE_OMEGA * v_x) + RNG.normal(0, 0.1, 300)

result = fit_custom(
    v_x, v_y,
    s_f_damped,
    v_p0=np.array([3.0, 0.3]),         # initial guesses for [A, gamma]
    v_fixed_params=np.array([TRUE_OMEGA]),  # omega is fixed
)

A_fit, gamma_fit = result.v_params
print(f"Fitted A     = {A_fit:.4f}  (true: {TRUE_A})")
print(f"Fitted gamma = {gamma_fit:.4f}  (true: {TRUE_GAMMA})")
print(result.summary())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(v_x, v_y, s=8, alpha=0.4, label="Noisy data")
axes[0].plot(v_x, A_fit * np.exp(-gamma_fit * v_x) * np.sin(TRUE_OMEGA * v_x),
             color="red", lw=2, label=f"Fit: A={A_fit:.2f}, γ={gamma_fit:.2f}")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title(f"Damped Oscillation  (R² = {result.metrics['r2']:.4f})")
axes[0].legend()

result.plot_residuals(ax=axes[1])
plt.tight_layout()
plt.show()


### 6.3 No Fixed Parameters — All Free

When no fixed parameters are needed, simply omit `v_fixed_params` (it defaults to `None`).  
All indices after `v_x[0]` are treated as free parameters.


In [ ]:
# Model: A * exp(b * x)  — both A and b are free
# v_x[0] = x data,  v_x[1] = A (free),  v_x[2] = b (free)
def s_f_exp(v3_x: np.ndarray) -> float:
    return v3_x[1] * np.exp(v3_x[2] * v3_x[0])

v_x = np.linspace(0, 2, 100)
v_y = 2.5 * np.exp(1.2 * v_x) + RNG.normal(0, 0.1, 100)

result = fit_custom(v_x, v_y, s_f_exp, v_p0=np.array([1.0, 1.0]))
A_fit, b_fit = result.v_params
print(f"Fitted A = {A_fit:.4f}  (true: 2.5)")
print(f"Fitted b = {b_fit:.4f}  (true: 1.2)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(v_x, v_y, s=20, alpha=0.5, label="Data")
ax.plot(v_x, A_fit * np.exp(b_fit * v_x), color="red", lw=2,
        label=f"Custom fit: {A_fit:.2f}·exp({b_fit:.2f}·x)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title(f"Custom Exponential (R² = {result.metrics['r2']:.4f})")
ax.legend()
plt.tight_layout()
plt.show()


---
## 7. Comparing Models on the Same Dataset

A common workflow: generate data from one underlying process, then compare how several models describe it using R² as the criterion.


In [ ]:
# Generate data from a power law
v_x = np.linspace(0.5, 6, 120)
v_y = 2.0 * v_x**1.8 + RNG.normal(0, 0.5, 120)

r_lin    = fit_linear(v_x, v_y)
r_poly2  = fit_polynomial(v_x, v_y, degree=2)
r_power  = fit_preset(v_x, v_y, model="power",  v_p0=np.array([2.0, 1.5]))
r_exp    = fit_preset(v_x, v_y, model="exponential", v_p0=np.array([1.0, 0.5, 0.0]))

models = [r_lin, r_poly2, r_power, r_exp]

print(f"{'Model':<20} {'R²':>8} {'RMSE':>10} {'MAE':>10}")
print("-" * 52)
for r in models:
    m = r.metrics
    print(f"{r.model_name:<20} {m['r2']:>8.4f} {m['rmse']:>10.4f} {m['mae']:>10.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(v_x, v_y, s=20, alpha=0.4, color="gray", label="Data")

def _make_poly2_y(r, v_x):
    a = r.v_params
    return a[0] + a[1]*v_x + a[2]*v_x**2

ax.plot(v_x, r_lin.v_params[0]  + r_lin.v_params[1] * v_x,
        lw=2, label=f"Linear   (R²={r_lin.metrics['r2']:.3f})")
ax.plot(v_x, _make_poly2_y(r_poly2, v_x),
        lw=2, label=f"Poly-2   (R²={r_poly2.metrics['r2']:.3f})")
ax.plot(v_x, r_power.v_params[0] * v_x**r_power.v_params[1],
        lw=2, label=f"Power    (R²={r_power.metrics['r2']:.3f})")
A_e, b_e, C_e = r_exp.v_params
ax.plot(v_x, A_e * np.exp(b_e * v_x) + C_e,
        lw=2, label=f"Exp      (R²={r_exp.metrics['r2']:.3f})")

ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Model Comparison on Power-Law Data")
ax.legend()
plt.tight_layout()
plt.show()
